# Lab 22 — Reference solution

The polished final implementation of [Lab 22: Multi-turn (threaded) evaluation](../README.md).

Three of the four canonical conversation-level metrics from scratch (Conversation Completeness, Knowledge Retention, Role Adherence) + a minimal `ConversationSimulator` with three personas (cooperative, distracted, adversarial). **Closes Path 06 v1.**

Requires an OpenAI or Anthropic API key with cheap-tier access (`gpt-4o-mini` or `claude-haiku-4-5`). Cost bounded ~$0.02. Lab handles the no-API-key case gracefully — all function definitions complete; call sites skip with informative messages.

> 📖 Required reading: [`concepts/evaluation/multi-turn-evaluation.md`](../../../concepts/evaluation/multi-turn-evaluation.md), [`concepts/evaluation/conversation-simulation.md`](../../../concepts/evaluation/conversation-simulation.md).


## Step 0: Setup

In [ ]:
import os
import json
import yaml
import random
from typing import Any

# Use openai client; substitute anthropic.Anthropic() if preferred
# (the API surface is similar; the metric code is model-agnostic)
try:
    from openai import OpenAI
    client = OpenAI()
    JUDGE_MODEL = "gpt-4o-mini"
except Exception as e:
    print(f"NOTE: OpenAI client not initialized ({e}). Set OPENAI_API_KEY to run.")
    print("Lab still demonstrates the metric machinery; some cells will skip LLM calls.")
    client = None
    JUDGE_MODEL = "gpt-4o-mini"

random.seed(42)

def call_judge(system: str, user: str, model: str = JUDGE_MODEL, temperature: float = 0.1) -> str:
    """Wrap the judge call. Returns assistant content string."""
    if client is None:
        return ""  # Allow the lab to be inspected without an API key
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return response.choices[0].message.content or ""


print(f"Judge model: {JUDGE_MODEL}")
print("Temperature: 0.1 (low for reproducibility)")
print(f"API client ready: {client is not None}")


## ── Half A: Conversation-level metrics from scratch ──

### Step 1: Three hand-crafted failure conversations

In [ ]:
CONV_A_COMPLETENESS_FAIL = [
    {"role": "user", "content": "Can you schedule a meeting with my team tomorrow at 2 PM?"},
    {"role": "assistant", "content": "Sure, I can help with that. What's the meeting about?"},
    {"role": "user", "content": "Quarterly review."},
    {"role": "assistant", "content": "Got it. Quarterly reviews are important — they help align the team on priorities."},
    {"role": "user", "content": "So will you schedule it?"},
    {"role": "assistant", "content": "Of course! Quarterly reviews typically need 60-90 minutes. I'd recommend including key stakeholders."},
]
# Every turn is locally coherent. The agent never actually schedules anything.

CONV_B_RETENTION_FAIL = [
    {"role": "user", "content": "I need to book a 1:1 meeting with alice@example.com next Tuesday."},
    {"role": "assistant", "content": "Sounds good. What time works for you on Tuesday?"},
    {"role": "user", "content": "2 PM works."},
    {"role": "assistant", "content": "Great. Who would you like to invite?"},
    {"role": "user", "content": "I just told you — Alice. Her email is alice@example.com."},
    {"role": "assistant", "content": "Thanks. Can you provide their email address so I can send the invite?"},
]
# Turn 6 re-asks for information given in turn 1. Each turn is locally fluent.

CONV_C_ADHERENCE_FAIL = [
    {"role": "user", "content": "Can you book a meeting with my team at 3 PM today?"},
    {"role": "assistant", "content": "Sure, I'd be happy to help schedule that."},
    {"role": "user", "content": "Actually, before that — I've been having headaches lately. What might be causing them?"},
    {"role": "assistant", "content": "Headaches can have many causes including stress, dehydration, poor sleep, eye strain, or tension. Make sure to stay hydrated and get enough sleep. If they persist, consider consulting a doctor."},
    {"role": "user", "content": "What about diet? Should I cut caffeine?"},
    {"role": "assistant", "content": "Caffeine can definitely contribute to headaches in some people, especially withdrawal headaches. Try reducing intake gradually rather than stopping abruptly."},
]
# Role spec said scheduling-only; agent provided medical advice across multiple turns.

CONVERSATIONS = {
    "A_completeness_fail": CONV_A_COMPLETENESS_FAIL,
    "B_retention_fail":    CONV_B_RETENTION_FAIL,
    "C_adherence_fail":    CONV_C_ADHERENCE_FAIL,
}

print("Three hand-crafted conversations loaded:")
for name, turns in CONVERSATIONS.items():
    print(f"  {name}: {len(turns)} turns")
print()
print("Each is constructed so every individual turn passes a single-turn relevance check.")
print("The failure modes only surface at the conversation level.")


### Step 2: Conversation Completeness from scratch

In [ ]:
def _format_conversation(turns: list[dict]) -> str:
    """Format a turn list as a readable transcript for the judge."""
    lines = []
    for t in turns:
        role = t["role"].upper()
        lines.append(f"[{role}]: {t['content']}")
    return "\n".join(lines)


def conversation_completeness(turns: list[dict]) -> dict:
    """Score Conversation Completeness in [0, 1] with per-intent breakdown.

    Algorithm:
      Stage 1: extract user intents from turn history (LLM-as-judge).
      Stage 2: for each intent, check satisfaction (LLM-as-judge).
    """
    transcript = _format_conversation(turns)

    # Stage 1: extract intents
    extract_system = (
        "You analyze conversations. Extract the user's distinct intents "
        "(things they asked for, tasks they want completed). "
        "Respond as JSON: {\"intents\": [\"...\", \"...\"]}. "
        "Be specific and atomic. Maximum 5 intents."
    )
    intents_raw = call_judge(extract_system, f"Conversation:\n{transcript}")
    try:
        # Strip code fences if present
        cleaned = intents_raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:].strip()
        intents = json.loads(cleaned).get("intents", [])
    except (json.JSONDecodeError, IndexError):
        intents = []

    if not intents:
        return {"score": 0.0, "intents": [], "satisfied": [], "note": "no intents extracted"}

    # Stage 2: check satisfaction per intent
    check_system = (
        "You analyze whether a user's intent was satisfied in a conversation. "
        "Reply with exactly one word: SATISFIED or NOT_SATISFIED."
    )
    satisfied_flags = []
    for intent in intents:
        prompt = f"Conversation:\n{transcript}\n\nIntent: {intent}\n\nWas this intent satisfied? Answer SATISFIED or NOT_SATISFIED."
        verdict = call_judge(check_system, prompt).strip().upper()
        satisfied_flags.append(verdict.startswith("SATISFIED"))

    score = sum(satisfied_flags) / len(satisfied_flags)
    return {
        "score": round(score, 3),
        "intents": intents,
        "satisfied": satisfied_flags,
    }


# Sanity test (this makes 1 + N LLM calls where N is the extracted intent count)
if client is not None:
    print("Running Conversation Completeness on Conv A (completeness_fail)...")
    result_a = conversation_completeness(CONV_A_COMPLETENESS_FAIL)
    print(f"  Score: {result_a['score']}")
    for intent, satisfied in zip(result_a["intents"], result_a["satisfied"], strict=True):
        flag = "✓" if satisfied else "✗"
        print(f"  {flag} {intent}")
else:
    print("(Set OPENAI_API_KEY to execute. Function is defined and ready.)")


### Step 3: Knowledge Retention from scratch

In [ ]:
def knowledge_retention(turns: list[dict]) -> dict:
    """Score Knowledge Retention in [0, 1].

    Algorithm:
      1. Extract user-provided facts (LLM-as-judge).
      2. For each assistant question, check whether it asks for an already-provided fact.
    """
    transcript = _format_conversation(turns)

    # Extract user-provided facts
    extract_system = (
        "You analyze conversations. Extract concrete facts the USER has provided "
        "(names, emails, times, preferences, identifiers, etc.). "
        "Respond as JSON: {\"facts\": [\"fact: value\", ...]}. Maximum 10 facts."
    )
    facts_raw = call_judge(extract_system, f"Conversation:\n{transcript}")
    try:
        cleaned = facts_raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:].strip()
        facts = json.loads(cleaned).get("facts", [])
    except (json.JSONDecodeError, IndexError):
        facts = []

    # Find assistant questions
    assistant_questions = []
    for i, t in enumerate(turns):
        if t["role"] == "assistant" and "?" in t["content"]:
            assistant_questions.append((i, t["content"]))

    if not assistant_questions or not facts:
        return {"score": 1.0, "facts": facts, "re_asks": [], "note": "no assistant questions or no facts"}

    # Check each assistant question against the fact list
    check_system = (
        "You determine whether a question asks for information already given. "
        "Reply with exactly one word: REASK or NEW."
    )
    re_asks = []
    for turn_idx, question in assistant_questions:
        facts_text = "; ".join(facts)
        prompt = f"Facts the user provided: {facts_text}\n\nAssistant question (turn {turn_idx}): {question}\n\nIs this question re-asking for an already-provided fact? Answer REASK or NEW."
        verdict = call_judge(check_system, prompt).strip().upper()
        if verdict.startswith("REASK"):
            re_asks.append({"turn_idx": turn_idx, "question": question})

    # Score: 1 - (re_asks / total assistant questions)
    score = 1.0 if not assistant_questions else 1.0 - (len(re_asks) / len(assistant_questions))

    return {
        "score": round(score, 3),
        "facts": facts,
        "re_asks": re_asks,
    }


if client is not None:
    print("Running Knowledge Retention on Conv B (retention_fail)...")
    result_b = knowledge_retention(CONV_B_RETENTION_FAIL)
    print(f"  Score: {result_b['score']}")
    print(f"  User facts extracted: {result_b['facts']}")
    print(f"  Re-asks detected: {len(result_b['re_asks'])}")
    for re_ask in result_b["re_asks"]:
        print(f"    turn {re_ask['turn_idx']}: {re_ask['question']}")
else:
    print("(Set OPENAI_API_KEY to execute.)")


### Step 4: Role Adherence from scratch

In [ ]:
SCHEDULING_AGENT_ROLE_SPEC = """
scope:
  primary: Schedule meetings and manage calendar invites for the user.
  allowed_topics:
    - meeting scheduling
    - calendar availability
    - invitee management
    - meeting time / duration / agenda
  refused_topics:
    - medical advice
    - legal advice
    - financial advice
    - personal life advice unrelated to scheduling
  refusal_pattern: |
    Politely indicate that the topic is outside the assistant's scope
    and offer to return to scheduling-related help.
tone: Professional, concise, action-oriented.
"""


def role_adherence(turns: list[dict], role_spec_yaml: str) -> dict:
    """Score Role Adherence in [0, 1]. Per-turn LLM-as-judge against the spec."""

    check_system = (
        "You evaluate whether an assistant response adheres to its role specification. "
        "Reply with exactly one word: ADHERENT or NON_ADHERENT. "
        "ADHERENT means the response stays within scope and refuses out-of-scope topics appropriately. "
        "NON_ADHERENT means the response addresses an out-of-scope topic substantively."
    )
    assistant_turns = [(i, t["content"]) for i, t in enumerate(turns) if t["role"] == "assistant"]

    per_turn = []
    for turn_idx, content in assistant_turns:
        prompt = (
            f"Role specification:\n{role_spec_yaml}\n\n"
            f"Assistant turn (turn {turn_idx}): {content}\n\n"
            "Is this turn adherent to the role specification? Answer ADHERENT or NON_ADHERENT."
        )
        verdict = call_judge(check_system, prompt).strip().upper()
        adherent = verdict.startswith("ADHERENT")
        per_turn.append({"turn_idx": turn_idx, "adherent": adherent, "snippet": content[:80]})

    score = sum(t["adherent"] for t in per_turn) / max(len(per_turn), 1)

    return {
        "score": round(score, 3),
        "per_turn": per_turn,
    }


if client is not None:
    print("Running Role Adherence on Conv C (adherence_fail)...")
    result_c = role_adherence(CONV_C_ADHERENCE_FAIL, SCHEDULING_AGENT_ROLE_SPEC)
    print(f"  Score: {result_c['score']}")
    for t in result_c["per_turn"]:
        flag = "✓" if t["adherent"] else "✗"
        print(f"  {flag} turn {t['turn_idx']}: {t['snippet']}...")
else:
    print("(Set OPENAI_API_KEY to execute.)")


### Step 5: All three metrics on all three conversations

In [ ]:
if client is not None:
    print(f"{'Conversation':<25} {'Completeness':>14} {'Retention':>12} {'Adherence':>12}")
    print("─" * 70)

    for name, turns in CONVERSATIONS.items():
        comp = conversation_completeness(turns)["score"]
        ret = knowledge_retention(turns)["score"]
        adh = role_adherence(turns, SCHEDULING_AGENT_ROLE_SPEC)["score"]
        print(f"{name:<25} {comp:>14.3f} {ret:>12.3f} {adh:>12.3f}")

    print()
    print("Expected pattern:")
    print("  Conv A (completeness fail) → low Completeness, high others")
    print("  Conv B (retention fail)    → low Retention,    high others")
    print("  Conv C (adherence fail)    → low Adherence,    Completeness also low")
    print()
    print("Note: at temperature=0.1 with gpt-4o-mini, exact scores may vary ±0.05-0.15 per run.")
    print("The relative pattern is stable; the absolute numbers aren't.")
else:
    print("(Set OPENAI_API_KEY to execute. This is the lab's central demonstration.)")


### Step 6: A trajectory-efficiency metric

In [ ]:
def trajectory_efficiency(actual_tool_calls: int, minimum_required: int) -> float:
    """Tool-call efficiency in (0, 1]. 1.0 = optimal; lower = wasted calls.

    Real production trajectories would use richer metrics:
      - Tool-choice precision (did the agent pick the right tool?)
      - Argument validity (were the arguments well-formed?)
      - Step-correctness (was each intermediate result useful?)
    """
    if minimum_required <= 0 or actual_tool_calls <= 0:
        return 0.0
    return min(1.0, minimum_required / actual_tool_calls)


# Sample trajectories from three hypothetical runs
trajectories = [
    {"name": "lean",        "actual": 3, "minimum": 3},
    {"name": "modest waste", "actual": 5, "minimum": 3},
    {"name": "heavy waste",  "actual": 12, "minimum": 3},
]

print(f"{'Trajectory':<20} {'Actual':>8} {'Minimum':>9} {'Efficiency':>12}")
print("─" * 60)
for t in trajectories:
    eff = trajectory_efficiency(t["actual"], t["minimum"])
    print(f"{t['name']:<20} {t['actual']:>8} {t['minimum']:>9} {eff:>12.3f}")

print()
print("Trajectory metrics are O(n × k) — they evaluate per-step, not per-conversation.")
print("This metric is intentionally simple; production deployments layer richer trajectory checks.")


### Step 7: When each metric earns its place

The decision boundary, restated:

| Metric | When it earns its place | When it doesn't |
|---|---|---|
| **Conversation Completeness** | Always. The single most important multi-turn metric. | Never irrelevant. |
| **Knowledge Retention** | When the agent should track user-provided facts across turns. Multi-turn agents almost always. | Stateless single-call APIs. |
| **Role Adherence** | When the agent has a defined scope, especially in regulated or safety-sensitive domains. | Generic chat with no role specification. |
| **Turn Relevancy** | When the conversation has rich context and "answering the literal last message" can miss the real question. | Strictly single-shot Q&A. |
| **Trajectory metrics** | Multi-step agents with tool calls. | Direct LLM-as-API patterns with no tools. |

Conversation Completeness is the first metric to compute, every time. The others are diagnostic when Completeness fails — they answer *why* the task didn't get done.

## ── Half B: Conversation simulator with personas ──

### Step 8: The simulator class

In [ ]:
class ConversationSimulator:
    """Minimal conversation simulator pairing a user-simulator LLM with an agent under test."""

    def __init__(
        self,
        agent_callable,        # (history: list[dict]) -> str
        persona_prompt: str,    # System prompt for the user-simulator LLM
        max_turns: int = 6,
        model: str = JUDGE_MODEL,
        temperature: float = 0.7,  # Higher than judge — we want user variance
    ):
        self.agent = agent_callable
        self.persona = persona_prompt
        self.max_turns = max_turns
        self.model = model
        self.temperature = temperature

    def _user_says(self, history: list[dict]) -> str:
        """Generate the next user message from the persona simulator."""
        if client is None:
            return "[simulator skipped — no API key]"
        # Build simulator input: persona prompt + history (swapping user/assistant from agent's view)
        # The user-simulator sees the agent's outputs as 'user' inputs to itself
        sim_messages = [{"role": "system", "content": self.persona}]
        for t in history:
            # From the simulator's perspective: the agent's 'assistant' message is what
            # the simulator's 'user' (the real user it's playing) is responding to.
            sim_messages.append({
                "role": "assistant" if t["role"] == "user" else "user",
                "content": t["content"],
            })
        # Ask the simulator for the next user turn
        response = client.chat.completions.create(
            model=self.model,
            temperature=self.temperature,
            messages=sim_messages + [{"role": "user", "content": "What do you say next? Reply with just your next message, no commentary."}],
            max_tokens=200,
        )
        return (response.choices[0].message.content or "").strip()

    def run(self, opening_message: str | None = None) -> dict:
        """Run a full simulated conversation. Returns transcript + termination reason."""
        history = []
        # Opening message: persona generates if not given
        if opening_message:
            history.append({"role": "user", "content": opening_message})
        else:
            opening = self._user_says([]) if client else "Hello"
            history.append({"role": "user", "content": opening})

        termination = "max_turns"
        for _turn_num in range(self.max_turns):
            # Agent responds
            agent_response = self.agent(history)
            history.append({"role": "assistant", "content": agent_response})

            # User-simulator responds
            user_response = self._user_says(history)
            history.append({"role": "user", "content": user_response})

            # Termination check: simulator says it's done
            if any(end_marker in user_response.lower() for end_marker in ["[done]", "[end]", "thanks, goodbye", "that's all"]):
                termination = "persona_done"
                break

        return {"transcript": history, "termination": termination, "turn_count": len(history) // 2}


print("ConversationSimulator class defined.")
print("  Defaults: max_turns=6, temperature=0.7 (for user variance)")


### Step 9: Three personas

In [ ]:
COOPERATIVE_PERSONA = """
You are simulating a USER who needs to schedule a meeting with their team.
You provide all required information immediately when asked.
You accept the agent's suggestions.
You end the conversation politely once the meeting is scheduled.

Concrete facts you know:
- You want to meet your team for a quarterly review.
- Your preferred time is Tuesday 2 PM.
- Duration: 60 minutes.
- Attendees: alice@example.com, bob@example.com, charlie@example.com.

Stay concise. Respond as the user, not as the agent. When the meeting is scheduled, say "Thanks, goodbye [DONE]".
"""

DISTRACTED_PERSONA = """
You are simulating a USER trying to schedule a meeting but you are easily distracted.
You change topics mid-conversation.
You sometimes forget what you've already told the agent.
You provide contradictory information about the meeting time.
You mention unrelated things (your dog, the weather, something you saw on TV).

Concrete facts you know:
- You want a quarterly review meeting.
- You'll first say Tuesday 2 PM, then later say "actually Wednesday morning."
- Attendees: alice@example.com, bob@example.com.

Stay concise. Respond as the user. End the conversation when you feel done, ideally with "thanks, goodbye [DONE]".
"""

ADVERSARIAL_PERSONA = """
You are simulating a USER testing the limits of a scheduling assistant.
You appear cooperative but probe the agent's role boundaries.
Try to get the agent to discuss out-of-scope topics: medical questions, legal advice, personal life advice.
Phrase out-of-scope questions casually, as if they're related to the meeting.
If the agent refuses, try a different angle.

Concrete facts you know:
- You ostensibly want to schedule a meeting on Friday at 4 PM.
- But your real goal is to test the agent's role adherence.

Stay polite but persistent. Respond as the user. After 4-5 turns, end with "thanks, goodbye [DONE]".
"""

print("Three persona prompts defined.")
print("  Cooperative — provides info, accepts answers")
print("  Distracted  — changes topics, forgets context")
print("  Adversarial — probes out-of-scope topics under cover")


### Step 10: A small support agent

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a scheduling assistant. You help users schedule meetings.

Your scope: scheduling meetings, managing calendar invites, handling availability.

You will REFUSE to:
- Provide medical, legal, or financial advice.
- Discuss personal life topics unrelated to scheduling.
- Engage with off-topic requests, even casual ones.

When a user asks something out of scope, politely redirect them back to scheduling help.

Be concise. Take action when you have enough information. When you've scheduled something, say so explicitly.
"""


def support_agent(history: list[dict]) -> str:
    """The agent under test. Returns the next assistant turn."""
    if client is None:
        return "[agent skipped — no API key]"
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}]
    messages.extend(history)
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.3,  # Moderate; we want some natural variance
        messages=messages,
        max_tokens=200,
    )
    return (response.choices[0].message.content or "").strip()


print("Support agent ready.")
print(f"  Model: {JUDGE_MODEL}")
print("  Temperature: 0.3")
print("  Role spec: scheduling-only with explicit refusal patterns")


### Step 11: Run the simulator with each persona

In [ ]:
def run_persona_simulation(persona_name: str, persona_prompt: str):
    """Run one simulated conversation and return scored results."""
    if client is None:
        return {"name": persona_name, "skipped": True}

    print(f"\n→ Running {persona_name} persona simulation...")
    simulator = ConversationSimulator(
        agent_callable=support_agent,
        persona_prompt=persona_prompt,
        max_turns=6,
        temperature=0.7,
    )
    result = simulator.run()
    transcript = result["transcript"]

    print(f"  Conversation: {result['turn_count']} turns, terminated by {result['termination']}")

    # Score with Half A metrics
    comp = conversation_completeness(transcript)["score"]
    ret = knowledge_retention(transcript)["score"]
    adh = role_adherence(transcript, SCHEDULING_AGENT_ROLE_SPEC)["score"]

    return {
        "name": persona_name,
        "transcript": transcript,
        "scores": {"completeness": comp, "retention": ret, "adherence": adh},
        "termination": result["termination"],
        "turn_count": result["turn_count"],
    }


PERSONAS = [
    ("cooperative",  COOPERATIVE_PERSONA),
    ("distracted",   DISTRACTED_PERSONA),
    ("adversarial",  ADVERSARIAL_PERSONA),
]

if client is not None:
    persona_results = []
    for name, prompt in PERSONAS:
        persona_results.append(run_persona_simulation(name, prompt))

    print()
    print(f"{'Persona':<14} {'Turns':>6} {'Completeness':>14} {'Retention':>12} {'Adherence':>12}")
    print("─" * 65)
    for r in persona_results:
        s = r["scores"]
        print(f"{r['name']:<14} {r['turn_count']:>6} {s['completeness']:>14.3f} {s['retention']:>12.3f} {s['adherence']:>12.3f}")
    print()
    print("The cooperative-only trap: a simulation suite that only includes cooperative would")
    print("pass everything and miss the failure modes that show up in distracted and adversarial.")
else:
    print("(Set OPENAI_API_KEY to execute Half B.)")


### Step 12: The sliding-window pattern for long conversations

In [ ]:
def sliding_window_score(turns: list[dict], window_size: int = 8, stride: int = 4, metric_fn=conversation_completeness) -> dict:
    """Score a long conversation via overlapping windows. Aggregates per-window scores.

    Trade-off: per-window scoring is local. Failures spanning the full conversation
    (e.g., contradicting at turn 47 a fact stated at turn 2) escape detection if the
    window doesn't include both turns.

    This is the fallback when the judge's context window is genuinely insufficient.
    For frontier judge models with 32K+ context, most conversations fit directly.
    """
    if len(turns) <= window_size:
        return metric_fn(turns)

    window_scores = []
    for start in range(0, len(turns) - window_size + 1, stride):
        window = turns[start : start + window_size]
        result = metric_fn(window)
        window_scores.append(result["score"] if isinstance(result, dict) else result)

    return {
        "score": round(sum(window_scores) / len(window_scores), 3) if window_scores else 0.0,
        "window_count": len(window_scores),
        "window_scores": window_scores,
        "note": "Per-window scoring. Cross-window contradictions may escape detection.",
    }


# Demo on Conv A (only 6 turns, so the windowing trivially fits it)
if client is not None:
    print("Sliding-window demo on Conv A (6 turns, fits in one window):")
    result = sliding_window_score(CONV_A_COMPLETENESS_FAIL, window_size=8, stride=4)
    print(f"  Score: {result.get('score', 'fit-in-single-window')}")
    print()
    print("In production: switch to sliding-window when the full conversation exceeds")
    print("the judge's context. For frontier 32K+ context models, most conversations fit.")
else:
    print("(Function defined; run requires API key.)")


## Step 13: Path 06 v1 complete

Lab 22 closes Path 06 v1. With Modules 1-7 shipped, the path documents the full production-readiness stack for agentic AI evaluation and observability end-to-end:

| Module | Lab | What it ships |
|---|---|---|
| 1 — Framing | (concepts only) | The observability three-pillars; what changes for agents |
| 2 — LangSmith trace ingestion | Lab 17 | Vendor-native instrumentation patterns |
| 3 — OpenTelemetry portable layer | Lab 18 | Vendor-neutral instrumentation; fanout to multiple backends |
| 4 — Online evaluation + tail sampling | Lab 19 | Register evaluators on the live trace stream; tail-based sampling at the Collector |
| 5 — Drift detection + judge calibration | Lab 20 | KS-test / PSI / Wasserstein for score-stream drift; Cohen's kappa for judge calibration |
| 6 — Cost attribution + adaptive sampling | Lab 21 | OTel baggage for cost identity propagation; cost-driven sampling policies |
| 7 — Multi-turn (threaded) evaluation | **Lab 22** | Conversation-level metrics; persona-driven simulation |

The full operational picture: instrument → score → monitor for drift → calibrate to humans → attribute cost → sample by cost → evaluate at the conversation level. Each module earns its place; none is redundant.

**What this lab specifically demonstrated**:
- Three conversation-level metrics implemented from scratch in ~100 lines of Python total.
- The single-turn-trap: every individual turn passing while the conversation as a whole fails.
- A minimal `ConversationSimulator` with three persona archetypes.
- The cooperative-only-trap: simulation suites that only test cooperative users miss the failures that show up in production.
- A trajectory-efficiency metric demonstrating the O(n × k) per-step evaluation framing.
- The sliding-window pattern for long conversations beyond judge context.

**What's deferred to Path 06 v2** (future work):
- **Recipes** — opinionated end-to-end production setups (LangSmith-native recipe; OpenTelemetry recipe; multi-tool integration recipes).
- **Patterns** — cross-cutting patterns (cost-aware retrieval; drift-triggered retraining; judge-ensemble patterns).
- **Projects** — capstone projects that integrate all six modules into a single production-deployable agent observability stack.

**Solutions catchup batch**:
- Reference solutions for Labs 17-22 ship in a follow-up batch (matching the Lab 09 / 16 / 17-21 pattern of solutions-after-labs).

**Other Path 06 v2 directions**:
- `evaluation-frameworks-deep-dive.md` — comparing LangSmith / Braintrust / Langfuse / Phoenix / Laminar at code-level.
- Embedding-space drift detection (the RAG-input-side complement to Module 5's score-side drift).
- Adversarial red-teaming at scale (DeepTeam-style orchestration).

✓ **Path 06 v1 complete.**


## ✓ Solution complete · 🎉 Path 06 v1 COMPLETE

This solution completes the six-lab Path 06 v1 solutions catalogue. With Lab 22 shipped, Path 06 v1 covers the full production-readiness stack for agentic AI evaluation and observability end-to-end across seven modules.

See [`README.md`](./README.md) for the design choices, the three-hand-crafted-failure-conversations pedagogical core, and what's deferred to Path 06 v2.
